# SNLI-Finetuned GPT-2 on MoRe
### This notebook cotains the results obtained from fine-tuning GPT-2 on SNLI and tsting it on the main test split of MoRe (i.e. the Hyponym Generalization Split)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/thesis_code


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/thesis_code


In [ ]:
import sys
sys.path.insert(
    0,
    '/content/drive/MyDrive/thesis_code/scripts'
)

In [ ]:
!pip install transformer-lens wget wandb ace_tools

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.4 MB/s eta 0:00:00
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=7ffb7c466c4b280bbadde905743595ae6643bdf0512d73e8bbd45dfad55ba730
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=a6d6f4d5ebfb496444cae35247b9bfc0595bbf8098c538319c592c9e670b082f
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built wget transformers-stream-generator


In [ ]:
import gpt_classification.evaluation_utils as evaluation_utils

In [ ]:
from gpt_classification.evaluation_utils import evaluate_model_on_dataset
from data.dataset_config import get_dataset_config
from gpt_classification.evaluation_utils import load_model

In [ ]:
#loading snli fine-tuned gpt2
snli_classifier = load_model(
    model_path="/content/drive/MyDrive/thesis_code/models/snli_classifier/gpt2_classifier.pth",
    dataset_name="snli"
)


Logging to: ./logs/process_and_train_20260818_204532.log


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda
Model loaded from: /content/drive/MyDrive/thesis_code/models/snli_classifier/gpt2_classifier.pth


In [ ]:
# loading MoRe
dataset_config = get_dataset_config("MoRe")
train_data, test_data, val_data, label_mapping = dataset_config.load_data(
    data_dir="/content/drive/MyDrive/thesis_code/data"
)
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"Labels: {list(label_mapping.keys())}")
example = test_data[0]
for k,v in example.items():
    print(f"{k:12s}: {v}")

MoRe dataset loaded successfully:
  Train = 458640 | Val = 105444 | Test = 468720
Train: 458640 | Val: 105444 | Test: 468720
Labels: ['entailment', 'neutral', 'contradiction']
id          : 1
premise     : All mammals moved across the field.
hypothesis  : All horses moved across the field.
gold_label  : entailment
category    : mammals
subcategory : mammals
hypernym    : mammals
hyponym     : horse
rule        : R1
operator    : all
monotonicity: downward


In [ ]:
# results on MoRe
results = evaluate_model_on_dataset(
    classifier=snli_classifier,
    dataset_name="MoRe",
    data_dir="/content/drive/MyDrive/thesis_code/data",
    split="test",
    save_csv=True,
    n_preview=5
)

results

Evaluating on 'MoRe' (test split)
MoRe dataset loaded successfully:
  Train = 458640 | Val = 105444 | Test = 468720
Dataset 'MoRe' loaded: Train=458640 | Val=105444 | Test=468720


100%|██████████| 14648/14648 [50:01<00:00,  4.88it/s]


Accuracy on MoRe (test split): 34.58%
Per-rule accuracy:
  R1: 3.88%
  R10: 80.91%
  R11: 33.33%
  R12: 96.32%
  R2: 3.30%
  R3: 67.77%
  R4: 66.36%
  R5: 6.94%
  R6: 1.81%
  R7: 6.15%
  R8: 3.76%
  R9: 44.42%
Predictions saved to ./results/eval_results/predictions_snli_classifier_on_MoRe_test.csv
Sample predictions:
1. Premise: All mammals moved across the field.
   Hypothesis: All horses moved across the field.
   Rule: R1
   Gold: 2 | Predicted: 0

2. Premise: All mammals rested near the trees.
   Hypothesis: All horses rested near the trees.
   Rule: R1
   Gold: 2 | Predicted: 0

3. Premise: All mammals gathered by the river.
   Hypothesis: All horses gathered by the river.
   Rule: R1
   Gold: 2 | Predicted: 0

4. Premise: All mammals searched for food.
   Hypothesis: All horses searched for food.
   Rule: R1
   Gold: 2 | Predicted: 0

5. Premise: All mammals walked together in a line.
   Hypothesis: All horses walked together in a line.
   Rule: R1
   Gold: 2 | Predicted: 0



{'overall_accuracy': 0.3457885304659712,
 'per_rule_accuracy': {'R1': 0.038812083973374295,
  'R2': 0.032974910394265235,
  'R3': 0.677726574500768,
  'R4': 0.663594470046083,
  'R5': 0.06938044034818229,
  'R6': 0.018125960061443933,
  'R7': 0.061495135688684074,
  'R8': 0.03763440860215054,
  'R9': 0.4441884280593958,
  'R10': 0.8090885816692268,
  'R11': 0.3332565284178187,
  'R12': 0.9631848438300051}}

In [ ]:
# loading SNLI
dataset_config = get_dataset_config("snli")
train_data, test_data, val_data, label_mapping = dataset_config.load_data(
    data_dir="/content/drive/MyDrive/thesis_code/data"
)
print(f"SNLI → Train: {len(train_data)} , Val: {len(val_data)}, Test: {len(test_data)}")

SNLI loaded: train=550152, val=10000, test=10000
SNLI → Train: 550152 | Val: 10000 | Test: 10000


In [ ]:
# Results on SNLI
acc, results = snli_classifier.evaluate(snli_classifier.dataloaders["test"], return_predictions=True)
print(f"\nOverall Accuracy on SNLI Test Set: {acc * 100:.2f}%")

100%|██████████| 307/307 [00:17<00:00, 17.86it/s]


Overall Accuracy on SNLI Test Set: 88.36%
